In [1]:

import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

forecast = pd.read_csv(filename)

forecast["HourUTC"] = pd.to_datetime(
    forecast["HourUTC"], utc=True
)

forecast["ForecastOriginUTC"] = pd.to_datetime(
    forecast["ForecastOriginUTC"], utc=True
)

forecast["HorizonHours"] = pd.to_numeric(
    forecast["HorizonHours"]
)

forecast["PredictedDemandMW"] = pd.to_numeric(
    forecast["PredictedDemandMW"]
)

# Validation
assert len(forecast) == 168
assert forecast["HourUTC"].is_unique
assert forecast["HourUTC"].is_monotonic_increasing
assert forecast["PredictedDemandMW"].notna().all()
assert np.isfinite(forecast["PredictedDemandMW"]).all()
assert (forecast["PredictedDemandMW"] >= 0).all()

expected_horizon = (
    forecast["HourUTC"] -
    forecast["ForecastOriginUTC"]
).dt.total_seconds() / 3600

assert (
    expected_horizon.to_numpy() ==
    forecast["HorizonHours"].to_numpy()
).all()

print("=== FORECAST VALIDATION ===")
print("Total Rows:", len(forecast))
print("Start:", forecast["HourUTC"].min())
print("End:", forecast["HourUTC"].max())
print("Average Forecast MW:",
      round(forecast["PredictedDemandMW"].mean(), 2))
print("Peak Forecast MW:",
      round(forecast["PredictedDemandMW"].max(), 2))

display(forecast.head())


Saving Forecast_Next_7_Days.csv to Forecast_Next_7_Days.csv
=== FORECAST VALIDATION ===
Total Rows: 168
Start: 2026-01-01 05:00:00+00:00
End: 2026-01-08 04:00:00+00:00
Average Forecast MW: 30036.71
Peak Forecast MW: 34093.89


,HourUTC,ForecastOriginUTC,HorizonHours,PredictedDemandMW,Model,LocalDateTime
0,2026-01-01 05:00:00+00:00,2026-01-01 04:00:00+00:00,1,27100.403347,Weekly Gradient Boosting,2026-01-01 00:00:00
1,2026-01-01 06:00:00+00:00,2026-01-01 04:00:00+00:00,2,27042.705384,Weekly Gradient Boosting,2026-01-01 01:00:00
2,2026-01-01 07:00:00+00:00,2026-01-01 04:00:00+00:00,3,27244.145811,Weekly Gradient Boosting,2026-01-01 02:00:00
3,2026-01-01 08:00:00+00:00,2026-01-01 04:00:00+00:00,4,27244.145811,Weekly Gradient Boosting,2026-01-01 03:00:00
4,2026-01-01 09:00:00+00:00,2026-01-01 04:00:00+00:00,5,27163.993676,Weekly Gradient Boosting,2026-01-01 04:00:00


In [3]:

# =====================================================
# QUEBEC ENERGY INTELLIGENCE
# STEP 11 - 14-DAY DEMAND FORECAST
# Backtest 2025 + Forecast January 2026
# =====================================================

import pandas as pd

# ==========================================
# LOAD ENERGY DATA - FIX CSV HEADERS
# ==========================================

filename = "Energy_ML_Hourly.csv"

# Read every row as data initially
data = pd.read_csv(
    filename,
    header=None,
    encoding="utf-8-sig"
)

# Expected 7 SQL export columns
columns = [
    "HourUTC",
    "FullDate",
    "HourOfDay",
    "DemandMW",
    "TemperatureC",
    "HumidityPercent",
    "PrecipitationMM"
]

if data.shape[1] != 7:
    raise ValueError(
        f"Expected 7 columns, found {data.shape[1]}"
    )

# Support both CSV formats:
# with headers and without headers
if str(data.iloc[0, 0]).strip() == "HourUTC":
    data = data.iloc[1:].copy()

# Set column names
data.columns = columns

# Always replace the previous df
df = data.copy()

df["HourUTC"] = pd.to_datetime(
    df["HourUTC"],
    utc=True,
    errors="raise"
)

df["DemandMW"] = pd.to_numeric(
    df["DemandMW"],
    errors="raise"
)

df = df.sort_values("HourUTC").reset_index(drop=True)

print("=== DATA LOADED SUCCESSFULLY ===")
print("Total Rows:", len(df))
print("Columns:", df.columns.tolist())
print("First Hour:", df["HourUTC"].min())
print("Last Hour:", df["HourUTC"].max())
print("Duplicate Hours:", df["HourUTC"].duplicated().sum())

display(df.head())


import io
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from google.colab import files
from IPython.display import display

HORIZON = 336  # 14 days * 24 hours

FEATURES = [
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "Lag_336",
    "Lag_504",
    "Lag_672"
]

# -----------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------

if "df" not in globals():
    print("Upload Energy_ML_Hourly.csv")
    uploaded = files.upload()
    df = pd.read_csv(
        io.BytesIO(next(iter(uploaded.values())))
    )

raw = df[["HourUTC", "DemandMW"]].copy()

raw["HourUTC"] = pd.to_datetime(
    raw["HourUTC"], utc=True
)

raw["DemandMW"] = pd.to_numeric(
    raw["DemandMW"], errors="raise"
)

raw = raw.sort_values("HourUTC")

if raw["HourUTC"].duplicated().any():
    raise ValueError("Duplicate UTC hours found")

if len(raw) < 50000:
    raise ValueError(
        "Historical dataset seems incomplete"
    )

hourly = raw.set_index("HourUTC").asfreq("h")

print("=== DATA SUMMARY ===")
print("Original rows:", len(raw))
print("Complete timeline:", len(hourly))
print("Missing demand hours:",
      hourly["DemandMW"].isna().sum())

# -----------------------------------------------------
# 2. FEATURE ENGINEERING
# -----------------------------------------------------

def create_features(data):

    result = data.copy()

    local = result.index.tz_convert(
        "America/Toronto"
    )

    result["Year"] = local.year
    result["Hour"] = local.hour
    result["DayOfWeek"] = local.dayofweek
    result["Month"] = local.month

    result["IsWeekend"] = (
        local.dayofweek >= 5
    ).astype(int)

    # Only lags available at a 14-day forecast origin
    result["Lag_336"] = (
        result["DemandMW"].shift(336)
    )

    result["Lag_504"] = (
        result["DemandMW"].shift(504)
    )

    result["Lag_672"] = (
        result["DemandMW"].shift(672)
    )

    return result


feature_data = create_features(hourly)

ready = feature_data.dropna(
    subset=["DemandMW"] + FEATURES
).copy()

# -----------------------------------------------------
# 3. TRAIN / TEST SPLIT
# -----------------------------------------------------

train = ready[ready["Year"] <= 2024]

print("\n=== TRAINING SUMMARY ===")
print("Training rows:", len(train))
print("Training end:", train.index.max())

# -----------------------------------------------------
# 4. CREATE MODEL
# -----------------------------------------------------

def create_model():
    return HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.06,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42
    )


model_14d = create_model()

model_14d.fit(
    train[FEATURES],
    train["DemandMW"]
)

# -----------------------------------------------------
# 5. 2025 ROLLING-ORIGIN BACKTEST
# Every simulation predicts up to 336 future hours
# -----------------------------------------------------

test_start = pd.Timestamp(
    "2025-01-01 05:00:00", tz="UTC"
)

test_end = pd.Timestamp(
    "2026-01-01 05:00:00", tz="UTC"
)

origin = test_start - pd.Timedelta(hours=1)

backtest_parts = []

while origin < test_end - pd.Timedelta(hours=1):

    end = min(
        origin + pd.Timedelta(hours=HORIZON),
        test_end - pd.Timedelta(hours=1)
    )

    block = ready.loc[
        (ready.index > origin)
        & (ready.index <= end)
        & (ready["Year"] == 2025)
    ].copy()

    if not block.empty:

        horizons = (
            (block.index - origin)
            / pd.Timedelta(hours=1)
        ).astype(int)

        # Safety: all lagged observations
        # must exist at or before forecast origin
        assert (
            block.index - pd.Timedelta(hours=336)
            <= origin
        ).all()

        predictions = model_14d.predict(
            block[FEATURES]
        )

        part = pd.DataFrame({
            "HourUTC": block.index,
            "ForecastOriginUTC": origin,
            "HorizonHours": horizons,
            "ActualDemandMW": block["DemandMW"].values,
            "PredictedDemandMW": predictions,
            "NaiveLag336": block["Lag_336"].values
        })

        backtest_parts.append(part)

    origin += pd.Timedelta(hours=HORIZON)

backtest = pd.concat(
    backtest_parts, ignore_index=True
)

backtest["Period"] = np.where(
    backtest["HorizonHours"] <= 168,
    "Days 1-7",
    "Days 8-14"
)

# -----------------------------------------------------
# 6. MODEL EVALUATION
# -----------------------------------------------------

def get_metrics(actual, predicted):

    return {
        "MAE": mean_absolute_error(
            actual, predicted
        ),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted)
    }


results = []

for label, subset in [
    ("Days 1-7", backtest[
        backtest["Period"] == "Days 1-7"
    ]),
    ("Days 8-14", backtest[
        backtest["Period"] == "Days 8-14"
    ]),
    ("Full 14 Days", backtest)
]:

    for model_name, pred_col in [
        ("Gradient Boosting", "PredictedDemandMW"),
        ("Naive Baseline Lag 336", "NaiveLag336")
    ]:

        scores = get_metrics(
            subset["ActualDemandMW"],
            subset[pred_col]
        )

        results.append({
            "Period": label,
            "Model": model_name,
            "TestHours": len(subset),
            **scores
        })

results_14d = pd.DataFrame(results)

print("\n=== 14-DAY BACKTEST RESULTS: 2025 ===")
display(results_14d.round(4))

# -----------------------------------------------------
# 7. RETRAIN USING ALL OBSERVED HISTORY
# -----------------------------------------------------

final_model_14d = create_model()

final_model_14d.fit(
    ready[FEATURES],
    ready["DemandMW"]
)

# -----------------------------------------------------
# 8. CREATE FUTURE 336 HOURS
# -----------------------------------------------------

cutoff = hourly["DemandMW"].last_valid_index()

future_hours = pd.date_range(
    start=cutoff + pd.Timedelta(hours=1),
    periods=HORIZON,
    freq="h",
    tz="UTC"
)

extended = hourly.reindex(
    hourly.index.union(future_hours)
)

future_features = create_features(extended).loc[
    future_hours, FEATURES
]

if future_features.isna().any().any():
    raise ValueError(
        "Missing lag features for future forecast"
    )

# -----------------------------------------------------
# 9. GENERATE FUTURE PREDICTIONS
# -----------------------------------------------------

predictions_14d = final_model_14d.predict(
    future_features
)

if not np.isfinite(predictions_14d).all():
    raise ValueError("Non-finite predictions found")

if (predictions_14d < 0).any():
    raise ValueError("Negative demand predictions found")

forecast_14d = pd.DataFrame({

    "HourUTC": future_hours.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),

    "ForecastOriginUTC": cutoff.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),

    "HorizonHours": np.arange(1, HORIZON + 1),

    "PredictedDemandMW": predictions_14d,

    "Model": "Biweekly Gradient Boosting",

    "LocalDateTime": future_hours.tz_convert(
        "America/Toronto"
    ).strftime("%Y-%m-%d %H:%M:%S")
})

# -----------------------------------------------------
# 10. VALIDATE AND EXPORT
# -----------------------------------------------------

assert len(forecast_14d) == 336
assert forecast_14d["HourUTC"].is_unique
assert forecast_14d["PredictedDemandMW"].notna().all()

print("\n=== FUTURE 14-DAY FORECAST ===")

print("Forecast origin:", cutoff)
print("Total hours:", len(forecast_14d))
print("Start UTC:", forecast_14d["HourUTC"].iloc[0])
print("End UTC:", forecast_14d["HourUTC"].iloc[-1])

print(
    "Average MW:",
    round(forecast_14d["PredictedDemandMW"].mean(), 2)
)

print(
    "Peak MW:",
    round(forecast_14d["PredictedDemandMW"].max(), 2)
)

display(forecast_14d.head())

filename = "Forecast_Next_14_Days.csv"

forecast_14d.to_csv(
    filename,
    index=False
)

files.download(filename)


=== DATA LOADED SUCCESSFULLY ===
Total Rows: 61306
Columns: ['HourUTC', 'FullDate', 'HourOfDay', 'DemandMW', 'TemperatureC', 'HumidityPercent', 'PrecipitationMM']
First Hour: 2019-01-01 06:00:00+00:00
Last Hour: 2026-01-01 04:00:00+00:00
Duplicate Hours: 0


,HourUTC,FullDate,HourOfDay,DemandMW,TemperatureC,HumidityPercent,PrecipitationMM
0,2019-01-01 06:00:00+00:00,2019-01-01,1,23762.55,0.8,98.0,1.3
1,2019-01-01 07:00:00+00:00,2019-01-01,2,23830.23,0.8,96.0,2.0
2,2019-01-01 08:00:00+00:00,2019-01-01,3,23608.07,0.9,98.0,3.1
3,2019-01-01 09:00:00+00:00,2019-01-01,4,23562.48,0.5,99.0,2.8
4,2019-01-01 10:00:00+00:00,2019-01-01,5,23546.16,0.7,94.0,2.9


=== DATA SUMMARY ===
Original rows: 61306
Complete timeline: 61367
Missing demand hours: 61

=== TRAINING SUMMARY ===
Training rows: 51699
Training end: 2025-01-01 04:00:00+00:00

=== 14-DAY BACKTEST RESULTS: 2025 ===


,Period,Model,TestHours,MAE,RMSE,R2
0,Days 1-7,Gradient Boosting,4390,1692.1410,2496.8724,0.8287
1,Days 1-7,Naive Baseline Lag 336,4390,2209.2145,3243.9051,0.7109
2,Days 8-14,Gradient Boosting,4362,1503.1154,2205.5310,0.8537
3,Days 8-14,Naive Baseline Lag 336,4362,1872.0225,2653.4266,0.7882
4,Full 14 Days,Gradient Boosting,8752,1597.9306,2356.1751,0.8406
5,Full 14 Days,Naive Baseline Lag 336,8752,2041.1579,2964.3493,0.7477



=== FUTURE 14-DAY FORECAST ===
Forecast origin: 2026-01-01 04:00:00+00:00
Total hours: 336
Start UTC: 2026-01-01 05:00:00
End UTC: 2026-01-15 04:00:00
Average MW: 30537.01
Peak MW: 35156.18


,HourUTC,ForecastOriginUTC,HorizonHours,PredictedDemandMW,Model,LocalDateTime
0,2026-01-01 05:00:00,2026-01-01 04:00:00,1,27600.500830,Biweekly Gradient Boosting,2026-01-01 00:00:00
1,2026-01-01 06:00:00,2026-01-01 04:00:00,2,27302.356538,Biweekly Gradient Boosting,2026-01-01 01:00:00
2,2026-01-01 07:00:00,2026-01-01 04:00:00,3,27170.339658,Biweekly Gradient Boosting,2026-01-01 02:00:00
3,2026-01-01 08:00:00,2026-01-01 04:00:00,4,27170.339658,Biweekly Gradient Boosting,2026-01-01 03:00:00
4,2026-01-01 09:00:00,2026-01-01 04:00:00,5,27310.137479,Biweekly Gradient Boosting,2026-01-01 04:00:00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:

# ==========================================
# STEP 21 - FINAL 14-DAY MODEL EVALUATION
# ==========================================

import pandas as pd

# Results created in Step 11
if "results_14d" not in globals():
    raise RuntimeError(
        "Run STEP 11 first to generate results_14d."
    )

print("=== MODEL BACKTEST COMPARISON: 2025 ===")

comparison = results_14d[
    [
        "Period",
        "Model",
        "TestHours",
        "MAE",
        "RMSE",
        "R2"
    ]
].copy()

display(comparison.round(4))

# Compare Gradient Boosting against baseline
pivot = comparison.pivot(
    index="Period",
    columns="Model",
    values="MAE"
)

print("\n=== MAE IMPROVEMENT VS BASELINE ===")

for period in ["Days 1-7", "Days 8-14", "Full 14 Days"]:

    model_mae = pivot.loc[
        period, "Gradient Boosting"
    ]

    baseline_mae = pivot.loc[
        period, "Naive Baseline Lag 336"
    ]

    improvement = (
        (baseline_mae - model_mae)
        / baseline_mae * 100
    )

    print(
        f"{period}: "
        f"Model MAE = {model_mae:.2f} MW | "
        f"Baseline MAE = {baseline_mae:.2f} MW | "
        f"Improvement = {improvement:.2f}%"
    )

print("\n=== BACKTEST COVERAGE ===")

print("Tested hours:", len(backtest))
print(
    "Unique forecast hours:",
    backtest["HourUTC"].nunique()
)
print(
    "Maximum horizon:",
    backtest["HorizonHours"].max()
)
print(
    "Forecast origins:",
    backtest["ForecastOriginUTC"].nunique()
)


=== MODEL BACKTEST COMPARISON: 2025 ===


,Period,Model,TestHours,MAE,RMSE,R2
0,Days 1-7,Gradient Boosting,4390,1692.1410,2496.8724,0.8287
1,Days 1-7,Naive Baseline Lag 336,4390,2209.2145,3243.9051,0.7109
2,Days 8-14,Gradient Boosting,4362,1503.1154,2205.5310,0.8537
3,Days 8-14,Naive Baseline Lag 336,4362,1872.0225,2653.4266,0.7882
4,Full 14 Days,Gradient Boosting,8752,1597.9306,2356.1751,0.8406
5,Full 14 Days,Naive Baseline Lag 336,8752,2041.1579,2964.3493,0.7477



=== MAE IMPROVEMENT VS BASELINE ===
Days 1-7: Model MAE = 1692.14 MW | Baseline MAE = 2209.21 MW | Improvement = 23.41%
Days 8-14: Model MAE = 1503.12 MW | Baseline MAE = 1872.02 MW | Improvement = 19.71%
Full 14 Days: Model MAE = 1597.93 MW | Baseline MAE = 2041.16 MW | Improvement = 21.71%

=== BACKTEST COVERAGE ===
Tested hours: 8752
Unique forecast hours: 8752
Maximum horizon: 336
Forecast origins: 27


In [2]:

import pandas as pd
import io
from google.colab import files

# Upload historical energy data
uploaded = files.upload()
filename = next(iter(uploaded))

columns = [
    "HourUTC",
    "FullDate",
    "HourOfDay",
    "DemandMW",
    "TemperatureC",
    "HumidityPercent",
    "PrecipitationMM"
]

# Read CSV including the first data row
df = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    header=None,
    encoding="utf-8-sig"
)

if df.shape[1] != 7:
    raise ValueError(
        f"Expected 7 columns, found {df.shape[1]}"
    )

# Remove header row only if the file has one
if str(df.iloc[0, 0]).strip() == "HourUTC":
    df = df.iloc[1:].copy()

df.columns = columns

df["HourUTC"] = pd.to_datetime(
    df["HourUTC"],
    utc=True,
    errors="raise"
)

df["DemandMW"] = pd.to_numeric(
    df["DemandMW"],
    errors="raise"
)

df = df.sort_values(
    "HourUTC"
).reset_index(drop=True)

print("=== DATA LOADED SUCCESSFULLY ===")
print("Total rows:", len(df))
print("First UTC:", df["HourUTC"].min())
print("Last UTC:", df["HourUTC"].max())
print(
    "Duplicate hours:",
    df["HourUTC"].duplicated().sum()
)

display(df.head())


Saving Energy_ML_Hourly.csv to Energy_ML_Hourly.csv
=== DATA LOADED SUCCESSFULLY ===
Total rows: 61306
First UTC: 2019-01-01 06:00:00+00:00
Last UTC: 2026-01-01 04:00:00+00:00
Duplicate hours: 0


,HourUTC,FullDate,HourOfDay,DemandMW,TemperatureC,HumidityPercent,PrecipitationMM
0,2019-01-01 06:00:00+00:00,2019-01-01,1,23762.55,0.8,98.0,1.3
1,2019-01-01 07:00:00+00:00,2019-01-01,2,23830.23,0.8,96.0,2.0
2,2019-01-01 08:00:00+00:00,2019-01-01,3,23608.07,0.9,98.0,3.1
3,2019-01-01 09:00:00+00:00,2019-01-01,4,23562.48,0.5,99.0,2.8
4,2019-01-01 10:00:00+00:00,2019-01-01,5,23546.16,0.7,94.0,2.9


In [3]:

# =============================================
# STEP 23 - 7-DAY ROLLING-ORIGIN BACKTEST
# Quebec Energy Intelligence
# =============================================

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from IPython.display import display

if "df" not in globals():
    raise RuntimeError(
        "Load Energy_ML_Hourly.csv into df first."
    )

# 1. Prepare historical data
data = df[["HourUTC", "DemandMW"]].copy()

data["HourUTC"] = pd.to_datetime(
    data["HourUTC"], utc=True
)

data["DemandMW"] = pd.to_numeric(
    data["DemandMW"], errors="raise"
)

if data["HourUTC"].duplicated().any():
    raise ValueError("Duplicate UTC timestamps")

hourly = (
    data.sort_values("HourUTC")
        .set_index("HourUTC")
        .asfreq("h")
)

# 2. Feature engineering
local = hourly.index.tz_convert("America/Toronto")

hourly["Year"] = local.year
hourly["Hour"] = local.hour
hourly["DayOfWeek"] = local.dayofweek
hourly["Month"] = local.month
hourly["IsWeekend"] = (
    local.dayofweek >= 5
).astype(int)

for lag in [168, 336, 504]:
    hourly[f"Lag_{lag}"] = (
        hourly["DemandMW"].shift(lag)
    )

features = [
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "Lag_168",
    "Lag_336",
    "Lag_504"
]

ready = hourly.dropna(
    subset=["DemandMW"] + features
).copy()

# 3. Train only on 2019-2024
train = ready[ready["Year"] <= 2024]

model_7d = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.06,
    max_leaf_nodes=31,
    min_samples_leaf=25,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

model_7d.fit(
    train[features],
    train["DemandMW"]
)

# 4. Rolling-origin backtest during 2025
test_start = pd.Timestamp(
    "2025-01-01 05:00:00", tz="UTC"
)

test_end = pd.Timestamp(
    "2026-01-01 05:00:00", tz="UTC"
)

origin = test_start - pd.Timedelta(hours=1)

blocks = []
origin_count = 0

while origin < test_end - pd.Timedelta(hours=1):

    end = min(
        origin + pd.Timedelta(hours=168),
        test_end - pd.Timedelta(hours=1)
    )

    block = ready.loc[
        (ready.index > origin)
        & (ready.index <= end)
        & (ready["Year"] == 2025)
    ].copy()

    if not block.empty:

        # No demand observed after origin
        # may be used in the lag features
        assert (
            block.index - pd.Timedelta(hours=168)
            <= origin
        ).all()

        predictions = model_7d.predict(
            block[features]
        )

        result = pd.DataFrame({
            "HourUTC": block.index,
            "ActualDemandMW": block["DemandMW"].values,
            "PredictedDemandMW": predictions,
            "BaselineMW": block["Lag_168"].values
        })

        blocks.append(result)
        origin_count += 1

    origin += pd.Timedelta(hours=168)

backtest_7d = pd.concat(
    blocks, ignore_index=True
)

# 5. Evaluate
def evaluate(name, prediction_column):

    actual = backtest_7d["ActualDemandMW"]
    predicted = backtest_7d[prediction_column]

    return {
        "TestYear": 2025,
        "ForecastDays": 7,
        "EvaluationPeriod": "Full 7 Days",
        "Model": name,
        "TestHours": len(backtest_7d),
        "MAE_MW": mean_absolute_error(
            actual, predicted
        ),
        "RMSE_MW": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted)
    }

results_7d_rolling = pd.DataFrame([
    evaluate(
        "Gradient Boosting",
        "PredictedDemandMW"
    ),
    evaluate(
        "Naive Baseline Lag 168",
        "BaselineMW"
    )
])

print("=== 7-DAY ROLLING BACKTEST: 2025 ===")
display(results_7d_rolling.round(4))

# 6. Compare with baseline
model_mae = results_7d_rolling.iloc[0]["MAE_MW"]
baseline_mae = results_7d_rolling.iloc[1]["MAE_MW"]

improvement = (
    (baseline_mae - model_mae)
    / baseline_mae * 100
)

print("\n=== EVALUATION SUMMARY ===")
print("Test hours:", len(backtest_7d))
print("Forecast origins:", origin_count)
print("Maximum horizon: 168 hours")
print(f"MAE improvement: {improvement:.2f}%")


=== 7-DAY ROLLING BACKTEST: 2025 ===


,TestYear,ForecastDays,EvaluationPeriod,Model,TestHours,MAE_MW,RMSE_MW,R2
0,2025,7,Full 7 Days,Gradient Boosting,8752,1572.3931,2363.6383,0.8396
1,2025,7,Full 7 Days,Naive Baseline Lag 168,8752,1844.9892,2716.4807,0.7881



=== EVALUATION SUMMARY ===
Test hours: 8752
Forecast origins: 53
Maximum horizon: 168 hours
MAE improvement: 14.77%
